In [2]:
import optuna_dashboard
optuna_dashboard.run_server(study.storage, host="localhost", port=8080)

C:\Users\Shiva\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NameError: name 'study' is not defined

In [ ]:
import numpy as np
import pandas as pd
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import QuantileTransformer, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
import optuna
from optuna.samplers import TPESampler

# Optimized Data Cleaning
class EfficientCleaner(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.num_cols = ['Atmospheric Density', 'Surface Temperature', 
                        'Gravity', 'Water Content', 'Mineral Abundance',
                        'Orbital Period', 'Proximity to Star', 
                        'Atmospheric Composition Index']
        self.cat_cols = ['Radiation Levels', 'Magnetic Field Strength']
        
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()
        # Numerical features
        for col in self.num_cols:
            X[col] = pd.to_numeric(X[col], errors='coerce')
            # Efficient outlier clipping
            q1, q9 = X[col].quantile([0.05, 0.95])
            X[col] = np.clip(X[col], q1, q9)
        
        # Categorical features
        for col in self.cat_cols:
            X[col] = pd.to_numeric(X[col].astype(str).str.extract(r'(\d+)', expand=False), errors='coerce')
        
        return X

# Fast Feature Engineering
class QuickFeatures(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()
        X['Habitable_Score'] = X['Atmospheric Composition Index'] * (1 - X['Radiation Levels']/10)
        X['Resource_Index'] = X['Mineral Abundance'] * np.log1p(X['Water Content'])
        return X

# Optimized Preprocessing
preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', IterativeImputer(max_iter=10, random_state=42)),
        ('scaler', QuantileTransformer(n_quantiles=100, output_distribution='normal'))
    ]), ['Atmospheric Density', 'Surface Temperature', 'Gravity',
         'Water Content', 'Mineral Abundance', 'Orbital Period',
         'Proximity to Star', 'Atmospheric Composition Index', 
         'Habitable_Score', 'Resource_Index']),
    
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), 
     ['Radiation Levels', 'Magnetic Field Strength'])
])

# XGBoost model with early stopping
model = XGBClassifier(
    tree_method='hist', 
    enable_categorical=False,
    eval_metric='merror',
    n_jobs=-1
)

# Full pipeline
pipeline = Pipeline([
    ('cleaner', EfficientCleaner()),
    ('features', QuickFeatures()),
    ('preprocessor', preprocessor),
    ('model', model)
])

# Optuna optimization
def objective(trial):
    params = {
        'model__learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'model__max_depth': trial.suggest_int('max_depth', 3, 9),
        'model__subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'model__colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'model__gamma': trial.suggest_float('gamma', 0, 0.5),
        'model__reg_alpha': trial.suggest_float('reg_alpha', 1e-6, 10, log=True),
        'model__reg_lambda': trial.suggest_float('reg_lambda', 1e-6, 10, log=True)
    }
    
    pipeline.set_params(**params)
    
    scores = []
    for train_idx, valid_idx in StratifiedKFold(n_splits=3, shuffle=True, random_state=42).split(X, y):
        X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
        X_valid, y_valid = X.iloc[valid_idx], y.iloc[valid_idx]
        
        pipeline.fit(
            X_train, y_train,
            model__early_stopping_rounds=20,
            model__eval_set=[(preprocessor.transform(X_valid)), y_valid],
            model__verbose=False
        )
        scores.append(pipeline.score(X_valid, y_valid))
    
    return np.mean(scores)

def main():
    # Load and prepare data
    df = pd.read_csv("train_data.csv")
    df['Prediction'] = pd.to_numeric(df['Prediction'], errors='coerce')
    df = df.dropna(subset=['Prediction'])
    df = df[df['Prediction'].between(0, 9)]
    
    global X, y
    X = df.drop('Prediction', axis=1)
    y = df['Prediction'].astype(int)
    
    # Optimization
    study = optuna.create_study(direction='maximize', sampler=TPESampler())
    study.optimize(objective, n_trials=50, timeout=3600)  # 1 hour timeout
    
    # Final training
    pipeline.set_params(**study.best_params)
    pipeline.fit(X, y)
    
    # Evaluation
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )
    print(f"Test Accuracy: {pipeline.score(X_test, y_test):.4f}")

if __name__ == "__main__":
    main()

KeyboardInterrupt: 

In [7]:
import numpy as np
import pandas as pd
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import QuantileTransformer, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
import xgboost
import optuna
from optuna.samplers import TPESampler

# Optimized Data Cleaning
class EfficientCleaner(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.num_cols = ['Atmospheric Density', 'Surface Temperature', 
                        'Gravity', 'Water Content', 'Mineral Abundance',
                        'Orbital Period', 'Proximity to Star', 
                        'Atmospheric Composition Index']
        self.cat_cols = ['Radiation Levels', 'Magnetic Field Strength']
        
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()
        # Numerical features
        for col in self.num_cols:
            X[col] = pd.to_numeric(X[col], errors='coerce')
            # Efficient outlier clipping
            q1, q9 = X[col].quantile([0.05, 0.95])
            X[col] = np.clip(X[col], q1, q9)
        
        # Categorical features
        for col in self.cat_cols:
            X[col] = pd.to_numeric(X[col].astype(str).str.extract(r'(\d+)', expand=False), errors='coerce')
        
        return X

# Fast Feature Engineering
class QuickFeatures(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()
        X['Habitable_Score'] = X['Atmospheric Composition Index'] * (1 - X['Radiation Levels']/10)
        X['Resource_Index'] = X['Mineral Abundance'] * np.log1p(X['Water Content'])
        X['atmospheric pressure'] = X['Gravity']*X['Atmospheric Density']
        X['Energy Recieved'] = X['Proximity to Star']*X['Orbital Period']
        X['Water state'] = X['Water Content']*X['Surface Temperature']
        X['Thermal Efficiency'] = X['Surface Temperature']/X['Proximity to Star']
        X['Atmospheric Retention'] = X['Atmospheric Density']/X['Gravity']
        X['Water phase indicator'] = X['Water Content']/X['Surface Temperature']
        X['Surface Temperature Polynomial'] = X['Surface Temperature']**2
        X['Gravity Polynomial'] = X['Gravity']**2
        X['Proximity to Star Polynomial'] = X['Proximity to Star']**2
        X['rf1'] = X['Water Content']/X['Mineral Abundance']
        X['rf2'] = X['Atmospheric Density']/X['Surface Temperature']
        X['rf3'] = X['Gravity']/X['Orbital Period']

        return X

# Optimized Preprocessing
preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', IterativeImputer(max_iter=10, random_state=42)),
        ('scaler', QuantileTransformer(n_quantiles=100, output_distribution='normal'))
    ]), ['Atmospheric Density', 'Surface Temperature', 'Gravity',
         'Water Content', 'Mineral Abundance', 'Orbital Period',
         'Proximity to Star', 'Atmospheric Composition Index', 
         'Habitable_Score', 'Resource_Index','atmospheric pressure','Energy Recieved',
         'Water state','Thermal Efficiency','Atmospheric Retention','Water phase indicator',
         'Surface Temperature Polynomial','Gravity Polynomial','Proximity to Star Polynomial',
         'rf1','rf2','rf3']),
    
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), 
     ['Radiation Levels', 'Magnetic Field Strength'])
])

# XGBoost model with early stopping
model = XGBClassifier(
    tree_method='hist', 
    enable_categorical=False,
    eval_metric='merror',
    n_jobs=-1,
    early_stopping_rounds=20,
    n_estimators=1000,  # Add a default number of estimators
)

# Full pipeline
pipeline = Pipeline([
    ('cleaner', EfficientCleaner()),
    ('features', QuickFeatures()),
    ('preprocessor', preprocessor),
    ('model', model)
])

# Optuna optimization
def objective(trial):
    params = {
        'model__learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'model__max_depth': trial.suggest_int('max_depth', 3, 9),
        'model__subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'model__colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'model__gamma': trial.suggest_float('gamma', 0, 0.5),
        'model__reg_alpha': trial.suggest_float('reg_alpha', 1e-6, 10, log=True),
        'model__reg_lambda': trial.suggest_float('reg_lambda', 1e-6, 10, log=True)
    }
    
    # Create a new pipeline with updated parameters
    temp_pipeline = Pipeline([
        ('cleaner', EfficientCleaner()),
        ('features', QuickFeatures()),
        ('preprocessor', preprocessor),
        ('model', XGBClassifier(
            tree_method='hist',
            enable_categorical=False,
            eval_metric='merror',
            n_jobs=-1,
            early_stopping_rounds=20,
            n_estimators=1000,
            **{k.replace('model__', ''): v for k, v in params.items()}
        ))
    ])
    
    scores = []
    for train_idx, valid_idx in StratifiedKFold(n_splits=3, shuffle=True, random_state=42).split(X, y):
        X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
        X_valid, y_valid = X.iloc[valid_idx], y.iloc[valid_idx]
        
        # First, fit and transform the training data
        X_train_processed = pipeline[:-1].fit_transform(X_train)
        # Then, transform validation data using the fitted preprocessor
        X_valid_processed = pipeline[:-1].transform(X_valid)
            # Only fit once with the proper parameters
        pipeline.fit(
                X_train, y_train,
                model__eval_set=[(X_valid_processed, y_valid)],
                model__verbose=False
            )
            
            # Get the score
        score = pipeline.score(X_valid, y_valid)
        scores.append(score)
        
        return np.mean(scores)

def main():
    # Load and prepare data
    df = pd.read_csv("train_data.csv")
    df['Prediction'] = pd.to_numeric(df['Prediction'], errors='coerce')
    df = df.dropna(subset=['Prediction'])
    df = df[df['Prediction'].between(0, 9)]
    
    global X, y
    X = df.drop('Prediction', axis=1)
    y = df['Prediction'].astype(int)
    
    # Create study with persistent storage
    storage = "sqlite:///optuna_study.db"
    study = optuna.create_study(
        direction="maximize",
        sampler=TPESampler(),
        storage=storage,
        study_name="cosmic_classifier_study",
        load_if_exists=True
    )
    
    # Optimization
    study.optimize(objective, n_trials=50, timeout=3600)  # 1 hour timeout
    
    # Final training
    pipeline.set_params(**study.best_params)
    pipeline.fit(X, y)
    
    # Evaluation
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )
    print(f"Test Accuracy: {pipeline.score(X_test, y_test):.4f}")

if __name__ == "__main__":
    main()

[I 2025-02-26 13:56:06,615] Using an existing study with name 'cosmic_classifier_study' instead of creating a new one.
C:\Users\Shiva\AppData\Roaming\Python\Python310\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
C:\Users\Shiva\AppData\Roaming\Python\Python310\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
C:\Users\Shiva\AppData\Roaming\Python\Python310\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
C:\Users\Shiva\AppData\Roaming\Python\Python310\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
[I 2025-02-26 13:56:33,194] Trial 56 finished with value: 0.8755990941170274 and parameters: {'le

ValueError: Invalid parameter 'learning_rate' for estimator Pipeline(steps=[('cleaner', EfficientCleaner()), ('features', QuickFeatures()),
                ('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   IterativeImputer(random_state=42)),
                                                                  ('scaler',
                                                                   QuantileTransformer(n_quantiles=100,
                                                                                       output_distribution='normal'))]),
                                                  ['Atmospheric Density',
                                                   'Surface Temperature',
                                                   'Gravity', 'Water Content',
                                                   'Mineral 